#### list 
<div>
bao secrets list <br>
/ # bao secrets list <br>
Path          Type         Accessor              Description <br>
----          ----         --------              ----------- <br>
cubbyhole/    cubbyhole    cubbyhole_d372195c    per-token private secret storage<br>
database/     database     database_4d85447d     n/a<br>
identity/     identity     identity_a3b6f171     identity store<br>
kv/           kv           kv_ed2a2ce8           n/a<br>
sys/          system       system_6b46f47d       system endpoints used for control, policy and debugging<br>
<br>
/ # bao kv get -field=user kv/db1<br>
ajay
<br>

bao login -method=userpass username=user1<br>

/ # bao --version
OpenBao v2.4.4 (4bfd70723d4f9b82be00e87b8c018ac661dd9b99), built 2025-11-24T19:54:48Z

In [18]:
import hvac

# Authentication (ensure Vault server is running and VAULT_TOKEN is set)
client = hvac.Client(
    url='http://127.0.0.1:8200',
    token='s.dpV8RRgprfGDkq0lYa1vv19W', # use environment variable in production
)
_version='v2'

if _version=='v1':
    if client.is_authenticated():
        # Reading a secret from the 'secret/data/my-secret-password' path
        resp = client.secrets.kv.v1.read_secret(
            path="db1",
            mount_point="secret"
        )
        password = resp["data"]["password"]
        # password = read_response['data']['data']['password']
        print(f"Retrieved password: {password}")
    else:
        print("Authentication failed.")
else:
    if client.is_authenticated():
        # Reading a secret from the 'secret/data/my-secret-password' path
        # read_response = client.secrets.kv.v2.read_secret_version(path='db1'
        #                                                         ,raise_on_deleted_version=True,   # keeps old behavior, no warning)
        # )

        read_response = client.secrets.kv.v2.read_secret_version(
            path="db1",
            mount_point="kv",
            raise_on_deleted_version=True
        )
                
        password = read_response['data']['data']['password']
        print(f"Retrieved password: {password}")
    else:
        print("Authentication failed.")

Retrieved password: callme123


In [21]:
import hvac
import os

# --- Configuration ---
# It is recommended to use environment variables for sensitive info
openbao_url = os.environ.get("OPENBAO_ADDR", "http://127.0.0.1:8200")
openbao_token = os.environ.get("OPENBAO_TOKEN", "s.dpV8RRgprfGDkq0lYa1vv19W")
secret_path = 'my-application/credentials'
secret_path = 'db1'
mount_point = 'kv' # The default mount point for kv-v2  secret
# mount_point = 'database_sql'
# --- Client Initialization ---
try:
    client = hvac.Client(url=openbao_url, token=openbao_token)

    if not client.is_authenticated():
        raise Exception("Failed to authenticate with OpenBao. Check OPENBAO_TOKEN.")

    print(f"Successfully authenticated with OpenBao at {openbao_url}")

    # --- Read Secret ---
    # For KV Version 2, use the 'read_secret_version' method.
    # The path provided to the method is relative to the mount point.
    read_response = client.secrets.kv.v2.read_secret_version(
        path=secret_path,
        mount_point=mount_point,
    )

    # Extract the secret data
    # secret_data = read_response['data']['data']
    secret_data = read_response['data']['data']
    # print(f"Secret data read from {mount_point}/data/{secret_path}:")
    print(f"Secret data read from {mount_point}/{secret_path}:")
    for key, value in secret_data.items():
        print(f"  {key}: {value}")

    # Example: Access a specific value
    # username = secret_data.get("username")
    # password = secret_data.get("password")


except hvac.exceptions.InvalidRequest as e:
    print(f"Error: Invalid request to OpenBao. Ensure the path '{mount_point}/data/{secret_path}' exists and is KV-v2: {e}")
except hvac.exceptions.Forbidden as e:
    print(f"Error: Forbidden access. Check policies and token permissions: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")



Successfully authenticated with OpenBao at http://127.0.0.1:8200
Secret data read from kv/db1:
  database_name: demo
  password: callme123
  port: 1433
  server: localhost
  user: ajay


C:\Users\ajsin\AppData\Local\Temp\ipykernel_32896\3333137376.py:24: DeprecationWarning: The raise_on_deleted_version parameter will change its default value to False in hvac v3.0.0. The current default of True will preserve previous behavior. To use the old behavior with no warning, explicitly set this value to True. See https://github.com/hvac/hvac/pull/907
  read_response = client.secrets.kv.v2.read_secret_version(


In [22]:
import hvac

client = hvac.Client(
    url="http://localhost:8200",
    token="s.dpV8RRgprfGDkq0lYa1vv19W"  # OpenBao token
)

# Check authentication
if not client.is_authenticated():
    raise Exception("Authentication failed")

# Read secret (KV v2)
response = client.secrets.kv.v2.read_secret_version(
    mount_point="kv",
    path="db1" 
)

print(response)
# Access secret values
secret_data = response["data"]["data"]
print(type(secret_data))
for k,v in secret_data.items():
    print(k,v)


{'request_id': 'c46e5dbb-0a23-a675-b3ed-0da74565689b', 'lease_id': '', 'renewable': False, 'lease_duration': 0, 'data': {'data': {'database_name': 'demo', 'password': 'callme123', 'port': '1433', 'server': 'localhost', 'user': 'ajay'}, 'metadata': {'created_time': '2026-01-19T10:42:23.524620558Z', 'custom_metadata': None, 'deletion_time': '', 'destroyed': False, 'version': 1}}, 'wrap_info': None, 'warnings': None, 'auth': None}
<class 'dict'>
database_name demo
password callme123
port 1433
server localhost
user ajay


C:\Users\ajsin\AppData\Local\Temp\ipykernel_32896\2433187658.py:13: DeprecationWarning: The raise_on_deleted_version parameter will change its default value to False in hvac v3.0.0. The current default of True will preserve previous behavior. To use the old behavior with no warning, explicitly set this value to True. See https://github.com/hvac/hvac/pull/907
  response = client.secrets.kv.v2.read_secret_version(


In [30]:
import hvac
import getpass # For securely entering passwords

_user ="user1"
try:
    client = hvac.Client(
        url="http://localhost:8200",
        # token="s.dpV8RRgprfGDkq0lYa1vv19W" , # OpenBao token 
        verify=False ,
    )

    login = client.auth.userpass.login(
            username=_user,
            password=_user ,
            mount_point="userpass",
    )

    # Ensure token is set for subsequent calls
    client.token = login["auth"]["client_token"]

    print("Authenticated:", client.is_authenticated())

    if client.is_authenticated():
        print("Successfully authenticated with Vault!")
        # Now you can interact with secrets (e.g., KV v2)
        # print(client.secrets.kv_v2.read_secret_version(path='myapp/config'))
    else:
        print("Authentication failed (check credentials or Vault setup).")

    print("2")
    # Read secret (KV v2)
    response = client.secrets.kv.v2.read_secret_version(
        mount_point="kv",
        path="db1" ,
        raise_on_deleted_version=True,   # keeps old behavior, no warning
    )

    print(response)
    # Access secret values
    secret_data = response["data"]["data"]
    print(type(secret_data))
    for k,v in secret_data.items():
        print(k,v)

except hvac.exceptions.InvalidRequest as e:
    print(f"Authentication Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Authenticated: True
Successfully authenticated with Vault!
2
{'request_id': '79e5b1fe-fe83-13a6-94dd-afe07d64578a', 'lease_id': '', 'renewable': False, 'lease_duration': 0, 'data': {'data': {'database_name': 'demo', 'password': 'callme123', 'port': '1433', 'server': 'localhost', 'user': 'ajay'}, 'metadata': {'created_time': '2026-01-19T10:42:23.524620558Z', 'custom_metadata': None, 'deletion_time': '', 'destroyed': False, 'version': 1}}, 'wrap_info': None, 'warnings': None, 'auth': None}
<class 'dict'>
database_name demo
password callme123
port 1433
server localhost
user ajay
